# PROYECTO DE GRADO II - UNAD
## Modelos adicionales recomendados por el asesor

Modelo predictivo de bajo peso al nacer en hijos de madres adolescentes.

**Recomendación atendida**

> Realizar dos modelos más de predicción: el primero eliminando las tres variables
> (Tiempo de Gestación, Número de consultas, Sexo). El segundo dejando la variable
> Sexo, eliminando (Tiempo de Gestación, Número de consultas).

**Configuraciones evaluadas**

| Configuración | Variables retiradas | Predictores |
|---|---|---|
| Modelo completo | Ninguna | 14 |
| Modelo A | T_GES, NUMCONSUL, SEXO | 11 |
| Modelo B | T_GES, NUMCONSUL | 12 |

El Modelo completo se incluye como referencia para que la comparación sea directa.
La diferencia entre el Modelo A y el Modelo B aísla el aporte de la variable SEXO,
porque ambos comparten el resto del conjunto de variables.

**Entrada**

El archivo `dataset_analitico_bajo_peso.csv`, producto de la Etapa 3. Este notebook
es independiente del notebook principal: reconstruye el protocolo completo y no
requiere ejecutar las etapas anteriores.

**Protocolo**

Idéntico al de las Etapas 5 y 6: partición estratificada 70/30 con semilla 42,
imputación y codificación dentro de un Pipeline, ponderación de clases balanceada,
búsqueda de hiperparámetros por validación cruzada de cinco particiones y evaluación
sobre el mismo conjunto de prueba. Mantener el protocolo es lo que permite atribuir
cualquier diferencia de desempeño al conjunto de variables y no al procedimiento.

## 1. Preparación

In [ ]:
# ------------------------------------------------------------
# PREPARACIÓN
# ------------------------------------------------------------
#
# Este notebook es independiente: reconstruye el protocolo de las
# Etapas 5 y 6 a partir del dataset analítico y entrena las tres
# configuraciones de variables en un solo procedimiento.
#
# Entrada: dataset_analitico_bajo_peso.csv (producto de la Etapa 3).
#
# Tiempo aproximado de ejecución: de cuatro a ocho minutos, según
# la máquina, por la búsqueda de hiperparámetros de nueve modelos.
#
# ------------------------------------------------------------

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_validate
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve
)


# ------------------------------------------------------------
# 1. PARÁMETROS DEL PROTOCOLO
# ------------------------------------------------------------
#
# Son los mismos valores de las Etapas 5 y 6. Mantenerlos es lo que
# permite comparar las configuraciones entre sí.

ARCHIVO_ENTRADA = "dataset_analitico_bajo_peso.csv"
CARPETA_SALIDA = "resultados_etapa_6b"

SEMILLA = 42
PROPORCION_PRUEBA = 0.30
NUMERO_FOLDS = 5
FRECUENCIA_MINIMA_CATEGORIA = 20

os.makedirs(CARPETA_SALIDA, exist_ok=True)

print("=" * 70)
print("MODELOS ADICIONALES RECOMENDADOS POR EL ASESOR")
print("=" * 70)

print(f"\nSemilla aleatoria: {SEMILLA}")
print(f"Proporción del conjunto de prueba: {PROPORCION_PRUEBA}")
print(f"Folds de validación cruzada: {NUMERO_FOLDS}")


# ------------------------------------------------------------
# 2. CARGA DEL DATASET ANALÍTICO
# ------------------------------------------------------------

df_modelado = pd.read_csv(ARCHIVO_ENTRADA)

print(f"\nRegistros: {df_modelado.shape[0]}")
print(f"Variables: {df_modelado.shape[1]}")
print(
    f"Casos de bajo peso: {int(df_modelado['BAJO_PESO'].sum())} "
    f"({df_modelado['BAJO_PESO'].mean() * 100:.2f} %)"
)


# ------------------------------------------------------------
# 3. CONJUNTO COMPLETO DE PREDICTORES
# ------------------------------------------------------------

variables_numericas = [
    "EDAD_MADRE", "EDAD_PADRE", "NUMCONSUL",
    "N_HIJOSV", "N_EMB", "T_GES"
]

variables_categoricas = [
    "EST_CIVM", "NIV_EDUM", "NIV_EDUP", "AREA_RES",
    "SEG_SOCIAL", "IDPUEBLOIN", "MUL_PARTO", "SEXO"
]

variables_numericas = [
    v for v in variables_numericas if v in df_modelado.columns
]
variables_categoricas = [
    v for v in variables_categoricas if v in df_modelado.columns
]

predictores = variables_numericas + variables_categoricas

X = df_modelado[predictores].copy()
y = df_modelado["BAJO_PESO"].astype(int).copy()

for variable in variables_categoricas:
    X[variable] = X[variable].astype("object")

print(f"\nPredictores del modelo completo: {len(predictores)}")


# ------------------------------------------------------------
# 4. PARTICIÓN ESTRATIFICADA
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=PROPORCION_PRUEBA,
    stratify=y,
    random_state=SEMILLA
)

print(f"\nEntrenamiento: {len(X_train)} registros ({int(y_train.sum())} casos)")
print(f"Prueba: {len(X_test)} registros ({int(y_test.sum())} casos)")


# ------------------------------------------------------------
# 5. ALGORITMOS Y REJILLAS
# ------------------------------------------------------------

configuracion_modelos = {

    "Regresión Logística": {
        "estimador": LogisticRegression(
            class_weight="balanced", max_iter=2000,
            solver="liblinear", random_state=SEMILLA
        ),
        "rejilla": {"modelo__C": [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]}
    },

    "Árbol de Decisión": {
        "estimador": DecisionTreeClassifier(
            class_weight="balanced", random_state=SEMILLA
        ),
        "rejilla": {
            "modelo__max_depth": [3, 4, 5, 6, 8],
            "modelo__min_samples_leaf": [20, 50, 100]
        }
    },

    "Random Forest": {
        "estimador": RandomForestClassifier(
            class_weight="balanced", n_estimators=300,
            n_jobs=-1, random_state=SEMILLA
        ),
        "rejilla": {
            "modelo__max_depth": [6, 10, None],
            "modelo__min_samples_leaf": [1, 20, 50]
        }
    }
}

validacion = StratifiedKFold(
    n_splits=NUMERO_FOLDS, shuffle=True, random_state=SEMILLA
)

metricas_validacion = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
    "accuracy": "accuracy"
}


# ------------------------------------------------------------
# 6. PREPROCESAMIENTO
# ------------------------------------------------------------

def construir_preprocesador(numericas, categoricas):

    transformador_numerico = Pipeline(steps=[
        ("imputador", SimpleImputer(strategy="median")),
        ("escalador", StandardScaler())
    ])

    transformador_categorico = Pipeline(steps=[
        ("imputador", SimpleImputer(strategy="most_frequent")),
        ("codificador", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=FRECUENCIA_MINIMA_CATEGORIA
        ))
    ])

    return ColumnTransformer(
        transformers=[
            ("numericas", transformador_numerico, numericas),
            ("categoricas", transformador_categorico, categoricas)
        ],
        remainder="drop"
    )


# ------------------------------------------------------------
# 7. FUNCIÓN DE MÉTRICAS
# ------------------------------------------------------------

def calcular_metricas(y_real, y_predicho, y_probabilidad):

    matriz = confusion_matrix(y_real, y_predicho, labels=[0, 1])
    vn, fp, fn, vp = matriz.ravel()

    especificidad = vn / (vn + fp) if (vn + fp) > 0 else np.nan
    vpn = vn / (vn + fn) if (vn + fn) > 0 else np.nan

    return {
        "Exactitud": accuracy_score(y_real, y_predicho),
        "Exactitud_balanceada": balanced_accuracy_score(y_real, y_predicho),
        "Sensibilidad": recall_score(y_real, y_predicho, zero_division=0),
        "Especificidad": especificidad,
        "Precision": precision_score(y_real, y_predicho, zero_division=0),
        "VPN": vpn,
        "F1": f1_score(y_real, y_predicho, zero_division=0),
        "ROC_AUC": roc_auc_score(y_real, y_probabilidad),
        "PR_AUC": average_precision_score(y_real, y_probabilidad),
        "VP": int(vp), "FP": int(fp), "VN": int(vn), "FN": int(fn)
    }


print("\nPreparación completada. Listo para entrenar las configuraciones.")


## 2. Entrenamiento, evaluación y comparación

In [ ]:
# ------------------------------------------------------------
# ETAPA 6B - MODELOS ADICIONALES RECOMENDADOS POR EL ASESOR
# ------------------------------------------------------------
#
# Recomendación recibida:
#
#   Modelo A: eliminar T_GES, NUMCONSUL y SEXO.
#   Modelo B: eliminar T_GES y NUMCONSUL, conservando SEXO.
#
# Ambas configuraciones se entrenan con el mismo protocolo del
# modelo completo: la misma partición estratificada (semilla 42),
# el mismo preprocesamiento, ponderación de clases balanceada,
# búsqueda de hiperparámetros por validación cruzada de cinco
# particiones y evaluación sobre el mismo conjunto de prueba.
#
# Mantener el protocolo es lo que permite atribuir cualquier
# diferencia de desempeño al conjunto de variables y no al
# procedimiento.
#
# Requiere haber ejecutado la celda de preparación anterior.
#
# ------------------------------------------------------------

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve, roc_auc_score


# ------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------

CARPETA_SALIDA = "resultados_etapa_6b"

os.makedirs(CARPETA_SALIDA, exist_ok=True)

print("=" * 70)
print("ETAPA 6B - MODELOS ADICIONALES RECOMENDADOS POR EL ASESOR")
print("=" * 70)

VARIABLES_RETIRADAS = {
    "Modelo A": ["T_GES", "NUMCONSUL", "SEXO"],
    "Modelo B": ["T_GES", "NUMCONSUL"]
}

ETIQUETAS = {
    "Modelo completo": "Catorce predictores (Etapas 5 y 6)",
    "Modelo A": "Sin edad gestacional, consultas prenatales ni sexo",
    "Modelo B": "Sin edad gestacional ni consultas prenatales"
}

print("\nConfiguraciones a evaluar:")
for nombre, retiradas in VARIABLES_RETIRADAS.items():
    print(f"\n{nombre}: {ETIQUETAS[nombre]}")
    print(f"  Variables retiradas: {', '.join(retiradas)}")


# ------------------------------------------------------------
# 2. CONJUNTOS DE PREDICTORES
# ------------------------------------------------------------

configuraciones = {
    "Modelo completo": {
        "numericas": list(variables_numericas),
        "categoricas": list(variables_categoricas)
    }
}

for nombre, retiradas in VARIABLES_RETIRADAS.items():
    configuraciones[nombre] = {
        "numericas": [v for v in variables_numericas if v not in retiradas],
        "categoricas": [v for v in variables_categoricas if v not in retiradas]
    }

filas_conjuntos = []

print("\n" + "=" * 70)
print("2. CONJUNTOS DE PREDICTORES")
print("=" * 70)

for nombre, cfg in configuraciones.items():

    predictores_cfg = cfg["numericas"] + cfg["categoricas"]
    cfg["predictores"] = predictores_cfg

    print(f"\n{nombre} — {len(predictores_cfg)} predictores")
    print(f"  Numéricas ({len(cfg['numericas'])}): {cfg['numericas']}")
    print(f"  Categóricas ({len(cfg['categoricas'])}): {cfg['categoricas']}")

    filas_conjuntos.append({
        "Configuracion": nombre,
        "Descripcion": ETIQUETAS[nombre],
        "Predictores": len(predictores_cfg),
        "Numericas": len(cfg["numericas"]),
        "Categoricas": len(cfg["categoricas"]),
        "Variables_retiradas": ", ".join(
            VARIABLES_RETIRADAS.get(nombre, [])
        ) or "Ninguna"
    })

tabla_conjuntos = pd.DataFrame(filas_conjuntos)

print("\nResumen de las configuraciones:")
display(tabla_conjuntos)

tabla_conjuntos.to_csv(
    os.path.join(CARPETA_SALIDA, "32_conjuntos_predictores.csv"),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 3. PREPROCESAMIENTO POR CONFIGURACIÓN
# ------------------------------------------------------------
#
# Se replica el preprocesamiento de la Etapa 5 sobre el subconjunto
# de columnas de cada configuración.

def construir_preprocesador(numericas, categoricas):

    transformador_numerico = Pipeline(steps=[
        ("imputador", SimpleImputer(strategy="median")),
        ("escalador", StandardScaler())
    ])

    transformador_categorico = Pipeline(steps=[
        ("imputador", SimpleImputer(strategy="most_frequent")),
        ("codificador", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=FRECUENCIA_MINIMA_CATEGORIA
        ))
    ])

    return ColumnTransformer(
        transformers=[
            ("numericas", transformador_numerico, numericas),
            ("categoricas", transformador_categorico, categoricas)
        ],
        remainder="drop"
    )


# ------------------------------------------------------------
# 4. ENTRENAMIENTO DE LAS TRES CONFIGURACIONES
# ------------------------------------------------------------
#
# La búsqueda de hiperparámetros se repite en cada configuración: al
# cambiar el conjunto de variables, el valor óptimo puede cambiar y
# conviene volver a buscarlo en lugar de arrastrar el anterior.
#
# Este es el paso lento del notebook.

print("\n" + "=" * 70)
print("4. ENTRENAMIENTO DE LAS TRES CONFIGURACIONES")
print("=" * 70)

modelos_adicionales = {}
filas_hiper_b = []

for nombre in ["Modelo completo", "Modelo A", "Modelo B"]:

    cfg = configuraciones[nombre]
    modelos_adicionales[nombre] = {}

    print(f"\n{'-' * 70}")
    print(f"{nombre}: {ETIQUETAS[nombre]}")
    print(f"{'-' * 70}")

    for algoritmo, configuracion in configuracion_modelos.items():

        flujo = Pipeline(steps=[
            ("preprocesamiento", construir_preprocesador(
                cfg["numericas"], cfg["categoricas"]
            )),
            ("modelo", clone(configuracion["estimador"]))
        ])

        busqueda = GridSearchCV(
            estimator=flujo,
            param_grid=configuracion["rejilla"],
            scoring="roc_auc",
            cv=validacion,
            n_jobs=-1,
            refit=True
        )

        busqueda.fit(X_train[cfg["predictores"]], y_train)

        modelos_adicionales[nombre][algoritmo] = busqueda.best_estimator_

        print(
            f"\n  {algoritmo}"
            f"\n    Mejores parámetros: {busqueda.best_params_}"
            f"\n    ROC-AUC en validación cruzada: {busqueda.best_score_:.4f}"
        )

        filas_hiper_b.append({
            "Configuracion": nombre,
            "Algoritmo": algoritmo,
            "Mejores_parametros": str(busqueda.best_params_),
            "ROC_AUC_validacion_cruzada": round(busqueda.best_score_, 4)
        })

tabla_hiper_b = pd.DataFrame(filas_hiper_b)

print("\nHiperparámetros seleccionados en cada configuración:")
display(tabla_hiper_b)

tabla_hiper_b.to_csv(
    os.path.join(CARPETA_SALIDA, "33_hiperparametros_adicionales.csv"),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 5. VALIDACIÓN CRUZADA DE LAS TRES CONFIGURACIONES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. VALIDACIÓN CRUZADA COMPARADA")
print("=" * 70)

todos_los_modelos = dict(modelos_adicionales)

filas_vc = []

for nombre, por_algoritmo in todos_los_modelos.items():

    cfg = configuraciones[nombre]

    for algoritmo, modelo in por_algoritmo.items():

        resultados = cross_validate(
            modelo,
            X_train[cfg["predictores"]],
            y_train,
            cv=validacion,
            scoring=metricas_validacion,
            n_jobs=-1
        )

        fila = {"Configuracion": nombre, "Algoritmo": algoritmo}

        for metrica in metricas_validacion:
            valores = resultados[f"test_{metrica}"]
            fila[f"{metrica}_media"] = round(valores.mean(), 4)
            fila[f"{metrica}_desv"] = round(valores.std(), 4)

        filas_vc.append(fila)

tabla_vc_b = pd.DataFrame(filas_vc)

print("\nValidación cruzada de las tres configuraciones:")
display(
    tabla_vc_b[[
        "Configuracion", "Algoritmo",
        "roc_auc_media", "roc_auc_desv",
        "recall_media", "f1_media"
    ]]
)

tabla_vc_b.to_csv(
    os.path.join(CARPETA_SALIDA, "34_validacion_cruzada_comparada.csv"),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 6. EVALUACIÓN EN EL CONJUNTO DE PRUEBA
# ------------------------------------------------------------
#
# Se reutiliza la función calcular_metricas definida en la Etapa 6,
# de modo que las cifras sean directamente comparables.

print("\n" + "=" * 70)
print("6. EVALUACIÓN EN EL CONJUNTO DE PRUEBA")
print("=" * 70)

filas_prueba = []
probabilidades_b = {}

for nombre, por_algoritmo in todos_los_modelos.items():

    cfg = configuraciones[nombre]

    for algoritmo, modelo in por_algoritmo.items():

        probabilidad = modelo.predict_proba(
            X_test[cfg["predictores"]]
        )[:, 1]

        prediccion = (probabilidad >= 0.5).astype(int)

        probabilidades_b[(nombre, algoritmo)] = probabilidad

        metricas = calcular_metricas(y_test, prediccion, probabilidad)
        metricas["Configuracion"] = nombre
        metricas["Algoritmo"] = algoritmo

        filas_prueba.append(metricas)

columnas_b = [
    "Configuracion", "Algoritmo", "ROC_AUC", "PR_AUC",
    "Sensibilidad", "Especificidad", "Precision", "F1",
    "Exactitud_balanceada", "VP", "FP", "VN", "FN"
]

tabla_prueba_b = pd.DataFrame(filas_prueba)[columnas_b].round(4)

print("\nMétricas en el conjunto de prueba con umbral de 0,50:")
display(tabla_prueba_b)

tabla_prueba_b.to_csv(
    os.path.join(CARPETA_SALIDA, "35_metricas_prueba_comparadas.csv"),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 7. MEJOR ALGORITMO DE CADA CONFIGURACIÓN
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("7. MEJOR ALGORITMO DE CADA CONFIGURACIÓN")
print("=" * 70)

mejores = (
    tabla_prueba_b
    .sort_values("ROC_AUC", ascending=False)
    .groupby("Configuracion", as_index=False)
    .first()
)

orden_config = ["Modelo completo", "Modelo A", "Modelo B"]

mejores["orden"] = mejores["Configuracion"].map(
    {c: i for i, c in enumerate(orden_config)}
)

mejores = mejores.sort_values("orden").drop(columns="orden")

referencia = float(
    mejores.loc[
        mejores["Configuracion"] == "Modelo completo", "ROC_AUC"
    ].iloc[0]
)

mejores["Diferencia_vs_completo"] = (
    mejores["ROC_AUC"] - referencia
).round(4)

mejores["Descripcion"] = mejores["Configuracion"].map(ETIQUETAS)

display(
    mejores[[
        "Configuracion", "Descripcion", "Algoritmo", "ROC_AUC",
        "PR_AUC", "Sensibilidad", "Especificidad", "F1",
        "Diferencia_vs_completo"
    ]]
)

mejores.to_csv(
    os.path.join(CARPETA_SALIDA, "36_mejor_algoritmo_por_configuracion.csv"),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 8. EFECTO DE CONSERVAR LA VARIABLE SEXO
# ------------------------------------------------------------
#
# La diferencia entre el Modelo A y el Modelo B aísla el aporte de
# SEXO, porque ambos comparten el resto del conjunto de variables.

print("\n" + "=" * 70)
print("8. EFECTO DE CONSERVAR LA VARIABLE SEXO")
print("=" * 70)

filas_sexo = []

for algoritmo in configuracion_modelos:

    fila_a = tabla_prueba_b[
        (tabla_prueba_b["Configuracion"] == "Modelo A")
        & (tabla_prueba_b["Algoritmo"] == algoritmo)
    ].iloc[0]

    fila_b = tabla_prueba_b[
        (tabla_prueba_b["Configuracion"] == "Modelo B")
        & (tabla_prueba_b["Algoritmo"] == algoritmo)
    ].iloc[0]

    filas_sexo.append({
        "Algoritmo": algoritmo,
        "ROC_AUC_sin_SEXO": fila_a["ROC_AUC"],
        "ROC_AUC_con_SEXO": fila_b["ROC_AUC"],
        "Diferencia": round(
            float(fila_b["ROC_AUC"]) - float(fila_a["ROC_AUC"]), 4
        ),
        "F1_sin_SEXO": fila_a["F1"],
        "F1_con_SEXO": fila_b["F1"]
    })

tabla_sexo = pd.DataFrame(filas_sexo)

print("\nAporte de la variable SEXO al desempeño:")
display(tabla_sexo)

tabla_sexo.to_csv(
    os.path.join(CARPETA_SALIDA, "37_efecto_variable_sexo.csv"),
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nDiferencia media de ROC-AUC atribuible a SEXO: "
    f"{tabla_sexo['Diferencia'].mean():+.4f}"
)


# ------------------------------------------------------------
# 9. CURVAS ROC DE LAS TRES CONFIGURACIONES
# ------------------------------------------------------------

plt.figure(figsize=(8, 6.5))

for _, fila in mejores.iterrows():

    clave = (fila["Configuracion"], fila["Algoritmo"])
    probabilidad = probabilidades_b[clave]

    fpr, tpr, _ = roc_curve(y_test, probabilidad)
    auc = roc_auc_score(y_test, probabilidad)

    plt.plot(
        fpr,
        tpr,
        label=(
            f"{fila['Configuracion']} — {fila['Algoritmo']} "
            f"(AUC = {auc:.4f})"
        )
    )

plt.plot(
    [0, 1], [0, 1],
    linestyle="--", color="gray", label="Clasificador aleatorio"
)

plt.xlabel("1 − Especificidad (tasa de falsos positivos)")
plt.ylabel("Sensibilidad (tasa de verdaderos positivos)")
plt.title(
    "Curvas ROC de las tres configuraciones en el conjunto de prueba"
)
plt.legend(loc="lower right", fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(CARPETA_SALIDA, "Figura_17_roc_configuraciones.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 10. COMPARACIÓN GRÁFICA DE MÉTRICAS
# ------------------------------------------------------------

metricas_grafico_b = ["ROC_AUC", "PR_AUC", "Sensibilidad", "Especificidad", "F1"]
etiquetas_grafico_b = ["ROC-AUC", "PR-AUC", "Sensibilidad", "Especificidad", "F1-score"]

posiciones = np.arange(len(metricas_grafico_b))
ancho = 0.26

plt.figure(figsize=(10, 6))

for indice, nombre in enumerate(orden_config):

    fila = mejores[mejores["Configuracion"] == nombre].iloc[0]
    valores = [float(fila[m]) for m in metricas_grafico_b]

    barras = plt.bar(
        posiciones + indice * ancho,
        valores,
        ancho,
        label=f"{nombre} ({fila['Algoritmo']})"
    )

    for barra, valor in zip(barras, valores):
        plt.text(
            barra.get_x() + barra.get_width() / 2,
            barra.get_height() + 0.012,
            f"{valor:.3f}".replace(".", ","),
            ha="center",
            fontsize=7.5
        )

plt.xticks(posiciones + ancho, etiquetas_grafico_b)
plt.ylim(0, 1.05)
plt.ylabel("Valor de la métrica")
plt.title(
    "Desempeño comparado de las tres configuraciones en el conjunto de prueba"
)
plt.legend(fontsize=9)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(CARPETA_SALIDA, "Figura_18_metricas_configuraciones.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 11. IMPORTANCIA DE VARIABLES EN LAS CONFIGURACIONES REDUCIDAS
# ------------------------------------------------------------
#
# Retiradas las variables dominantes, interesa saber qué sostiene la
# predicción en los modelos reducidos.

print("\n" + "=" * 70)
print("11. IMPORTANCIA POR PERMUTACIÓN EN LOS MODELOS REDUCIDOS")
print("=" * 70)

filas_importancia_b = []

for nombre in ["Modelo A", "Modelo B"]:

    fila_mejor = mejores[mejores["Configuracion"] == nombre].iloc[0]
    algoritmo = fila_mejor["Algoritmo"]
    cfg = configuraciones[nombre]

    resultado = permutation_importance(
        modelos_adicionales[nombre][algoritmo],
        X_test[cfg["predictores"]],
        y_test,
        scoring="roc_auc",
        n_repeats=10,
        random_state=SEMILLA,
        n_jobs=-1
    )

    parcial = pd.DataFrame({
        "Configuracion": nombre,
        "Algoritmo": algoritmo,
        "Variable": cfg["predictores"],
        "Caida_media_ROC_AUC": resultado.importances_mean.round(4),
        "Desviacion": resultado.importances_std.round(4)
    }).sort_values("Caida_media_ROC_AUC", ascending=False)

    print(f"\n{nombre} ({algoritmo}) — cinco variables principales:")
    display(parcial.head(5))

    filas_importancia_b.append(parcial)

tabla_importancia_b = pd.concat(filas_importancia_b, ignore_index=True)

tabla_importancia_b.to_csv(
    os.path.join(CARPETA_SALIDA, "38_importancia_modelos_reducidos.csv"),
    index=False,
    encoding="utf-8-sig"
)

fig, ejes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True)

for eje, nombre in zip(ejes, ["Modelo A", "Modelo B"]):

    parcial = tabla_importancia_b[
        tabla_importancia_b["Configuracion"] == nombre
    ].sort_values("Caida_media_ROC_AUC")

    eje.barh(
        parcial["Variable"],
        parcial["Caida_media_ROC_AUC"],
        xerr=parcial["Desviacion"],
        capsize=3
    )

    eje.set_title(f"{nombre}", fontsize=11)
    eje.grid(axis="x", alpha=0.3)
    eje.axvline(0, color="gray", linewidth=0.8)

ejes[0].set_xlabel("Caída media del ROC-AUC al permutar la variable")
ejes[1].set_xlabel("Caída media del ROC-AUC al permutar la variable")

fig.suptitle(
    "Importancia por permutación en las configuraciones reducidas"
)

fig.tight_layout()

fig.savefig(
    os.path.join(CARPETA_SALIDA, "Figura_19_importancia_reducidos.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 12. AJUSTE DEL UMBRAL EN LOS MODELOS REDUCIDOS
# ------------------------------------------------------------
#
# Con una capacidad de discriminación menor, el punto de operación
# adquiere más peso: conviene revisar qué sensibilidad se alcanza y
# a qué costo en falsos positivos.

print("\n" + "=" * 70)
print("12. PUNTOS DE OPERACIÓN DE LOS MODELOS REDUCIDOS")
print("=" * 70)

rejilla_b = np.arange(0.05, 0.96, 0.01)

filas_operacion = []

for nombre in orden_config:

    fila_mejor = mejores[mejores["Configuracion"] == nombre].iloc[0]
    algoritmo = fila_mejor["Algoritmo"]
    cfg = configuraciones[nombre]

    modelo = todos_los_modelos[nombre][algoritmo]

    probabilidad_train = modelo.predict_proba(
        X_train[cfg["predictores"]]
    )[:, 1]

    mejor_j, mejor_umbral = -1.0, 0.5

    for umbral in rejilla_b:

        prediccion = (probabilidad_train >= umbral).astype(int)
        matriz = confusion_matrix(y_train, prediccion, labels=[0, 1])
        vn, fp, fn, vp = matriz.ravel()

        sens = vp / (vp + fn) if (vp + fn) else 0.0
        esp = vn / (vn + fp) if (vn + fp) else 0.0

        if sens + esp - 1 > mejor_j:
            mejor_j, mejor_umbral = sens + esp - 1, umbral

    probabilidad = probabilidades_b[(nombre, algoritmo)]
    prediccion = (probabilidad >= mejor_umbral).astype(int)
    metricas = calcular_metricas(y_test, prediccion, probabilidad)

    filas_operacion.append({
        "Configuracion": nombre,
        "Algoritmo": algoritmo,
        "Umbral_Youden": round(float(mejor_umbral), 2),
        "Sensibilidad": round(metricas["Sensibilidad"], 4),
        "Especificidad": round(metricas["Especificidad"], 4),
        "Precision": round(metricas["Precision"], 4),
        "VP": metricas["VP"],
        "FP": metricas["FP"],
        "FN": metricas["FN"]
    })

tabla_operacion = pd.DataFrame(filas_operacion)

print("\nPunto de operación de máximo índice de Youden:")
display(tabla_operacion)

tabla_operacion.to_csv(
    os.path.join(CARPETA_SALIDA, "39_puntos_operacion.csv"),
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 13. RESUMEN DE LA ETAPA 6B
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RESUMEN DE LA ETAPA 6B")
print("=" * 70)

for nombre in orden_config:

    fila = mejores[mejores["Configuracion"] == nombre].iloc[0]

    print(
        f"\n{nombre} ({len(configuraciones[nombre]['predictores'])} predictores)"
        f"\n  Mejor algoritmo: {fila['Algoritmo']}"
        f"\n  ROC-AUC en prueba: {fila['ROC_AUC']:.4f}"
        f"\n  Sensibilidad: {fila['Sensibilidad']:.4f}"
        f"\n  Especificidad: {fila['Especificidad']:.4f}"
        f"\n  F1-score: {fila['F1']:.4f}"
    )

print("\nArchivos generados:")

for archivo in sorted(os.listdir(CARPETA_SALIDA)):
    print("-", archivo)

print("\nETAPA 6B FINALIZADA.")


## 3. Descarga de resultados (opcional, en Google Colab)

Descomenta las líneas siguientes para descargar la carpeta con las tablas y figuras.

In [ ]:
#!zip -r resultados_etapa_6b.zip resultados_etapa_6b
#from google.colab import files
#files.download('resultados_etapa_6b.zip')
